# 01 - Database Setup, Ingestion, and Raw Data Export
This notebook ingests daily OHLCV bars and writes canonical raw data files to data/raw for downstream feature engineering.

In [22]:
from pathlib import Path
import logging
import sys

import pandas as pd
from sqlalchemy import text

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_settings
from src.database import create_all_tables, get_engine, get_session_factory, session_scope
from src.data_loader import fetch_ohlcv_yfinance, load_price_bars_to_dataframe, upsert_price_bars
from src.logging_utils import configure_logging

configure_logging(logging.INFO)
logger = logging.getLogger("notebook.01_database_setup")
logger.info("Notebook initialization complete.")

2026-04-02 18:07:21,801 | INFO | notebook.01_database_setup | Notebook initialization complete.


In [23]:
import os
from sqlalchemy.exc import OperationalError

logger.info("Step 1/4: Initializing database engine and creating tables.")
settings = get_settings()
engine = get_engine(settings.db_url)
session_factory = get_session_factory(engine)

try:
    create_all_tables(engine)
    logger.info("Table creation/check complete on primary DB.")
except OperationalError as exc:
    sqlite_path = PROJECT_ROOT / "data" / "processed" / "quant_alpha_local.db"
    sqlite_path.parent.mkdir(parents=True, exist_ok=True)
    sqlite_url = f"sqlite+pysqlite:///{sqlite_path.as_posix()}"
    os.environ["DATABASE_URL"] = sqlite_url
    
    settings = get_settings()
    engine = get_engine(settings.db_url)
    session_factory = get_session_factory(engine)
    create_all_tables(engine)
    
    logger.warning("PostgreSQL unavailable; switched to SQLite fallback.")
    logger.warning("Original error type: %s", exc.__class__.__name__)

print(f'Connected DB: {settings.db_url}')

2026-04-02 18:07:21,850 | INFO | notebook.01_database_setup | Step 1/4: Initializing database engine and creating tables.


2026-04-02 18:07:21,908 | INFO | notebook.01_database_setup | Table creation/check complete on primary DB.


Connected DB: sqlite+pysqlite:///c:/Users/kenne/OneDrive/Desktop/Quant_Projects_2026/Cross-Sectional-Machine-Learning-Alpha-for-Equities/data/processed/quant_alpha_local.db


In [24]:
logger.info("Step 2/4: Downloading OHLCV bars from data provider.")
tickers = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL']
bars_df = fetch_ohlcv_yfinance(
    tickers=tickers,
    start_date=settings.start_date,
    end_date=settings.end_date,
)

raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
raw_parquet_path = raw_dir / "ohlcv_raw.parquet"
raw_csv_path = raw_dir / "ohlcv_raw.csv"

parquet_written = False
try:
    bars_df.to_parquet(raw_parquet_path, index=False)
    parquet_written = True
except ImportError:
    logger.warning("Parquet engine not available; exporting raw data to CSV only.")

bars_df.to_csv(raw_csv_path, index=False)
if parquet_written:
    logger.info("Raw data exported to %s and %s", raw_parquet_path, raw_csv_path)
else:
    logger.info("Raw data exported to %s", raw_csv_path)

print("Raw csv:", raw_csv_path)
if parquet_written:
    print("Raw parquet:", raw_parquet_path)
print(bars_df.head())
print('Rows fetched:', len(bars_df))

2026-04-02 18:07:21,927 | INFO | notebook.01_database_setup | Step 2/4: Downloading OHLCV bars from data provider.


2026-04-02 18:07:22,386 | WARNING | notebook.01_database_setup | Parquet engine not available; exporting raw data to CSV only.
2026-04-02 18:07:22,487 | INFO | notebook.01_database_setup | Raw data exported to c:\Users\kenne\OneDrive\Desktop\Quant_Projects_2026\Cross-Sectional-Machine-Learning-Alpha-for-Equities\data\raw\ohlcv_raw.csv


Raw csv: c:\Users\kenne\OneDrive\Desktop\Quant_Projects_2026\Cross-Sectional-Machine-Learning-Alpha-for-Equities\data\raw\ohlcv_raw.csv
Price        date ticker       open       high        low      close  \
0      2015-01-02   AAPL  24.671147  24.682222  23.776350  24.214890   
1      2015-01-02   AMZN  15.629000  15.737500  15.348000  15.426000   
2      2015-01-02  GOOGL  26.411708  26.570398  26.177643  26.260460   
3      2015-01-02   MSFT  39.682632  40.328983  39.580578  39.767677   
4      2015-01-02   NVDA   0.482985   0.486584   0.475307   0.482985   

Price     volume  
0      212818400  
1       55664000  
2       26480000  
3       27913900  
4      113680000  
Rows fetched: 13825


In [25]:
logger.info("Step 3/4: Upserting raw bars into SQL table for optional DB-backed workflows.")
with session_scope(session_factory) as session:
    upsert_price_bars(session, bars_df)

with session_scope(session_factory) as session:
    count = session.execute(text('SELECT COUNT(*) FROM price_bars')).scalar_one()

print('Rows now in SQL table price_bars:', count)

2026-04-02 18:07:22,506 | INFO | notebook.01_database_setup | Step 3/4: Upserting raw bars into SQL table for optional DB-backed workflows.


Rows now in SQL table price_bars: 13825


In [26]:
logger.info("Step 4/4: Verifying SQL round-trip and previewing stored rows.")
with session_scope(session_factory) as session:
    sql_df = load_price_bars_to_dataframe(session)

sql_df.head()

2026-04-02 18:07:24,775 | INFO | notebook.01_database_setup | Step 4/4: Verifying SQL round-trip and previewing stored rows.


,date,ticker,open,high,low,close,volume
0,2015-01-02,AAPL,24.671147,24.682222,23.776350,24.214890,212818400.0
1,2015-01-02,AMZN,15.629000,15.737500,15.348000,15.426000,55664000.0
2,2015-01-02,GOOGL,26.411708,26.570398,26.177643,26.260460,26480000.0
3,2015-01-02,MSFT,39.682632,40.328983,39.580578,39.767677,27913900.0
4,2015-01-02,NVDA,0.482985,0.486584,0.475307,0.482985,113680000.0
